In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print("PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True")

PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True


# Explainability Evaluation - Stability

## Testing the Attention based Explainability and SHAP on Stability

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
/content/bachelor-thesis
Already up to date.
/content/bachelor-thesis
Repo exists: True


In [4]:
# Imports
import os
import numpy as np
import pandas as pd
import shap
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import StabilityEvaluation
from explainability.shap_wrappers import AnoFormerAnomalyScoreWrapper, TranADAnomalyScoreWrapper
from explainability.sample_selection import load_samples_from_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [5]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

STABILITY_DIR = RESULTS_DIR / "evaluation" / "stability"
STABILITY_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [7]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
# We need the Train dataset because the SHAP background is from the training set. We also need the mean and std of the training set to normalize the test set.
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

# We need the Test dataset because the chosen samples for evaluation are from the test set
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [8]:
def make_single_ecg_loader(ecg):
    if isinstance(ecg, np.ndarray):
        ecg = torch.tensor(ecg, dtype=torch.float32)

    if ecg.ndim == 2:
        ecg = ecg.unsqueeze(0)

    dummy_label = torch.zeros(ecg.shape[0], dtype=torch.long)
    dataset = TensorDataset(ecg, dummy_label)

    return DataLoader(dataset, batch_size=1, shuffle=False)

In [9]:
# Load fixed evaluation samples

EvaluationSamplePath = (
    RESULTS_DIR
    / "evaluation"
    / "evaluation_samples.npz"
)

EvaluationSamples = load_samples_from_dataset(
    sample_path=EvaluationSamplePath,
    test_dataset=test_dataset,
    device=None,
)

print("Number of evaluation samples:", len(EvaluationSamples))

for sample in EvaluationSamples:
    print(
        sample["index"],
        sample["case_type"],
        sample["label"],
        sample["ecg"].shape,
    )

Number of evaluation samples: 20
2154 TN 0 torch.Size([1, 1000, 12])
1647 TN 0 torch.Size([1, 1000, 12])
161 TN 0 torch.Size([1, 1000, 12])
1394 TN 0 torch.Size([1, 1000, 12])
929 TN 0 torch.Size([1, 1000, 12])
911 TN 0 torch.Size([1, 1000, 12])
1528 TN 0 torch.Size([1, 1000, 12])
193 TN 0 torch.Size([1, 1000, 12])
383 TN 0 torch.Size([1, 1000, 12])
1878 TN 0 torch.Size([1, 1000, 12])
1134 TP 1 torch.Size([1, 1000, 12])
1854 TP 1 torch.Size([1, 1000, 12])
102 TP 1 torch.Size([1, 1000, 12])
1310 TP 1 torch.Size([1, 1000, 12])
1462 TP 1 torch.Size([1, 1000, 12])
1041 TP 1 torch.Size([1, 1000, 12])
56 TP 1 torch.Size([1, 1000, 12])
653 TP 1 torch.Size([1, 1000, 12])
910 TP 1 torch.Size([1, 1000, 12])
1332 TP 1 torch.Size([1, 1000, 12])


In [10]:
# Load previously calculated Anomaly Thresholds
# these were calculated based on the evaluation set

filepath = MODEL_DIR / "chosen_thresholds.json"

with open(filepath, "r") as f:
    thresholds = json.load(f)

AnoFormerChosenThreshold = thresholds["anoformer"]
TranADChosenThreshold = thresholds["tranad"]

In [11]:
# Create the Evaluator

StabilityEvaluator = StabilityEvaluation(
    noise_ratio=0.01,
    epsilon=1e-8
)

# AnoFormer Evaluation

In [12]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)


Model loaded from: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [13]:
def anoformer_score_function(ecg):
    """
    Return the AnoFormer anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = AnoFormer.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=AnoFormerChosenThreshold,
        NumMaskRuns=5,
        Seed=RANDOM_SEED,
    )

    return scores[0]

### Attention Evaluation

In [14]:
OriginalImportanceValues = []
NoisyImportanceValues = [] # List of Lists, each ECG has a list of the importance values of its noisy variants

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    original_importance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

    noisy_importances = []

    for _ in range(5):
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)

        noisy_importance = AnoFormerExplainer.get_attention_importance(
            TimeSeriesWindow=noisy_ecg,
            BlockIndex=0,
            HeadIndex=None,
        )

        noisy_importances.append(noisy_importance)

    OriginalImportanceValues.append(original_importance)
    NoisyImportanceValues.append(noisy_importances)

In [15]:
# Evaluate

AnoFormerAttentionStabilityResults = pd.DataFrame(
    StabilityEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        original_importance_values=OriginalImportanceValues,
        noisy_importance_values=NoisyImportanceValues,
        model_name="AnoFormer",
        explanation_name="Attention",
    )
)

AnoFormerAttentionStabilityResults

,Model,Explanation,Index,Case Type,Label,Stability Score,Stability Std,Number of Noisy Variants
0,AnoFormer,Attention,2154,TN,0,0.999998,7.904811e-07,5
1,AnoFormer,Attention,1647,TN,0,0.999994,2.562631e-06,5
2,AnoFormer,Attention,161,TN,0,0.999995,1.786743e-06,5
3,AnoFormer,Attention,1394,TN,0,0.999997,1.192429e-06,5
4,AnoFormer,Attention,929,TN,0,0.999994,2.557441e-06,5
5,AnoFormer,Attention,911,TN,0,0.999992,7.403280e-06,5
6,AnoFormer,Attention,1528,TN,0,0.999994,4.739252e-06,5
7,AnoFormer,Attention,193,TN,0,0.999991,5.063242e-06,5
8,AnoFormer,Attention,383,TN,0,0.999997,1.474885e-06,5
9,AnoFormer,Attention,1878,TN,0,0.999997,1.823297e-06,5


In [16]:
AnoFormerAttentionStabilitySummary = (
    AnoFormerAttentionStabilityResults
    .groupby(["Model", "Explanation", "Case Type"])["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerAttentionStabilitySummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,Attention,TN,0.999995,0.000002,10
1,AnoFormer,Attention,TP,0.999946,0.000100,10


In [17]:
AnoFormerAttentionStabilityResults.to_csv(
    STABILITY_DIR / "anoformer_attention_stability_results.csv",
    index=False,
)

AnoFormerAttentionStabilitySummary.to_csv(
    STABILITY_DIR / "anoformer_attention_stability_summary.csv",
    index=False,
)

## SHAP Evaluation

In [18]:
# Load the background indices used for SHAP explanations

background_indices = np.load(
    SHAP_RESULTS_DIR / "small_background_indices.npy"
)

background_ECGs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

In [19]:
# Load the original shap values

AnoFormerSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "anoformer"
    / "anoformer_evaluation_shap_values.npy"
)

print("AnoFormer SHAP values:", AnoFormerSHAPValues.shape)

AnoFormer SHAP values: (20, 1000, 12)


In [20]:
# because SHAP takes very long, choose a smaller subset to evaluate

num_samples_per_case = 5         # 5 TP, 5 TN
num_noisy_variants = 2

TNPositions = [
    position
    for position, sample in enumerate(EvaluationSamples)
    if sample["case_type"] == "TN"
]

TPPositions = [
    position
    for position, sample in enumerate(EvaluationSamples)
    if sample["case_type"] == "TP"
]

SHAPStabilityPositions = (
    TNPositions[:num_samples_per_case]
    + TPPositions[:num_samples_per_case]
)

SHAPStabilitySamples = [
    EvaluationSamples[position]
    for position in SHAPStabilityPositions
]

print("Number of SHAP stability samples:", len(SHAPStabilitySamples))
print("Positions:", SHAPStabilityPositions)

Number of SHAP stability samples: 10
Positions: [0, 1, 2, 3, 4, 10, 11, 12, 13, 14]


In [21]:
# Create the same Anomaly Score Wrapper for AnoFormer that was used during SHAP explanations

AnoFormerScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,
    num_masks=1,
    seed=RANDOM_SEED,
).to(DEVICE)

AnoFormerScoreWrapper.eval()

# create Explainer
AnoFormerSHAPExplainer = shap.GradientExplainer(
    AnoFormerScoreWrapper,
    background_ECGs,
    batch_size=1,
)

In [22]:
AnoFormerSHAPOriginalImportanceValues = []
AnoFormerSHAPNoisyImportanceValues = []

for position, sample in zip(SHAPStabilityPositions, SHAPStabilitySamples):
    ecg = sample["ecg"].to(AnoFormer.Device)

    original_shap_values = AnoFormerSHAPValues[position]
    original_importance = np.abs(original_shap_values).sum(axis=1)

    noisy_importances = []

    for _ in range(num_noisy_variants):

        # create a noisy variant and calculate it's shap values
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)
        noisy_shap_values = AnoFormerSHAPExplainer.shap_values(noisy_ecg)

        if isinstance(noisy_shap_values, list):
            noisy_shap_values = noisy_shap_values[0]

        noisy_shap_values = np.asarray(noisy_shap_values)

        if noisy_shap_values.ndim == 4:
            noisy_shap_values = noisy_shap_values[..., 0]

        noisy_shap_values = noisy_shap_values[0]

        noisy_importance = np.abs(noisy_shap_values).sum(axis=1)

        noisy_importances.append(noisy_importance)
        # Clear CUDA cache after each noisy variant calculation to free up memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    AnoFormerSHAPOriginalImportanceValues.append(original_importance)
    AnoFormerSHAPNoisyImportanceValues.append(noisy_importances)

    print(
        "Computed AnoFormer SHAP stability importances for index:",
        sample["index"],
        "case:",
        sample["case_type"]
    )
    # Clear CUDA cache after processing each sample as well
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Computed AnoFormer SHAP stability importances for index: 2154 case: TN
Computed AnoFormer SHAP stability importances for index: 1647 case: TN
Computed AnoFormer SHAP stability importances for index: 161 case: TN
Computed AnoFormer SHAP stability importances for index: 1394 case: TN
Computed AnoFormer SHAP stability importances for index: 929 case: TN
Computed AnoFormer SHAP stability importances for index: 1134 case: TP
Computed AnoFormer SHAP stability importances for index: 1854 case: TP
Computed AnoFormer SHAP stability importances for index: 102 case: TP
Computed AnoFormer SHAP stability importances for index: 1310 case: TP
Computed AnoFormer SHAP stability importances for index: 1462 case: TP


In [23]:
# Evaluate

AnoFormerSHAPStabilityList = (
    StabilityEvaluator.evaluate_samples(
        samples=SHAPStabilitySamples,
        original_importance_values=AnoFormerSHAPOriginalImportanceValues,
        noisy_importance_values=AnoFormerSHAPNoisyImportanceValues,
        model_name="AnoFormer",
        explanation_name="SHAP",
    )
)

AnoFormerSHAPStabilityResults = pd.DataFrame(
    AnoFormerSHAPStabilityList
)

AnoFormerSHAPStabilityResults

,Model,Explanation,Index,Case Type,Label,Stability Score,Stability Std,Number of Noisy Variants
0,AnoFormer,SHAP,2154,TN,0,0.996831,0.001058,2
1,AnoFormer,SHAP,1647,TN,0,0.993570,0.004234,2
2,AnoFormer,SHAP,161,TN,0,0.986976,0.002669,2
3,AnoFormer,SHAP,1394,TN,0,0.995906,0.000235,2
4,AnoFormer,SHAP,929,TN,0,0.992590,0.002692,2
5,AnoFormer,SHAP,1134,TP,1,0.975116,0.016438,2
6,AnoFormer,SHAP,1854,TP,1,0.994743,0.002173,2
7,AnoFormer,SHAP,102,TP,1,0.986811,0.003274,2
8,AnoFormer,SHAP,1310,TP,1,0.997408,0.001862,2
9,AnoFormer,SHAP,1462,TP,1,0.966748,0.011007,2


In [24]:
AnoFormerSHAPStabilitySummary = (
    AnoFormerSHAPStabilityResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerSHAPStabilitySummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,SHAP,TN,0.993175,0.003865,5
1,AnoFormer,SHAP,TP,0.984165,0.013034,5


In [25]:
# Save

AnoFormerSHAPStabilityResults.to_csv(
    STABILITY_DIR / "anoformer_shap_stability_results.csv",
    index=False,
)

AnoFormerSHAPStabilitySummary.to_csv(
    STABILITY_DIR / "anoformer_shap_stability_summary.csv",
    index=False,
)

# TranAD

In [26]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)


TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [27]:
def tranad_score_function(ecg):
    """
    Return the TranAD anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = TranAD.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=TranADChosenThreshold,
    )

    return scores[0]

## Attention Evaluation

In [28]:
OriginalImportanceValues = []
NoisyImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    original_importance = TranADExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        LayerIndex=0,
        HeadIndex=None,
    )

    noisy_importances = []

    for _ in range(5):
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)

        noisy_importance = TranADExplainer.get_attention_importance(
            TimeSeriesWindow=ecg,
            LayerIndex=0,
            HeadIndex=None,
        )

        noisy_importances.append(noisy_importance)

    OriginalImportanceValues.append(original_importance)
    NoisyImportanceValues.append(noisy_importances)

In [29]:
# Evaluate

TranADAttentionStabilityResults = pd.DataFrame(
    StabilityEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        original_importance_values=OriginalImportanceValues,
        noisy_importance_values=NoisyImportanceValues,
        model_name="TranAD",
        explanation_name="Attention",
    )
)

TranADAttentionStabilityResults

,Model,Explanation,Index,Case Type,Label,Stability Score,Stability Std,Number of Noisy Variants
0,TranAD,Attention,2154,TN,0,1.0,0.0,5
1,TranAD,Attention,1647,TN,0,1.0,0.0,5
2,TranAD,Attention,161,TN,0,1.0,0.0,5
3,TranAD,Attention,1394,TN,0,1.0,0.0,5
4,TranAD,Attention,929,TN,0,1.0,0.0,5
5,TranAD,Attention,911,TN,0,1.0,0.0,5
6,TranAD,Attention,1528,TN,0,1.0,0.0,5
7,TranAD,Attention,193,TN,0,1.0,0.0,5
8,TranAD,Attention,383,TN,0,1.0,0.0,5
9,TranAD,Attention,1878,TN,0,1.0,0.0,5


In [30]:
TranADAttentionStabilitySummary = (
    TranADAttentionStabilityResults
    .groupby(["Model", "Explanation", "Case Type"])["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADAttentionStabilitySummary

,Model,Explanation,Case Type,mean,std,count
0,TranAD,Attention,TN,1.0,6.505313e-16,10
1,TranAD,Attention,TP,1.0,7.797950e-16,10


In [31]:
# Save the Results
TranADAttentionStabilityResults.to_csv(
    STABILITY_DIR / "tranad_attention_stability_results.csv",
    index=False,
)

TranADAttentionStabilitySummary.to_csv(
    STABILITY_DIR / "tranad_attention_stability_summary.csv",
    index=False,
)

## SHAP Evaluation

In [32]:
# Load the original shap values

TranADSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "tranad"
    / "tranad_evaluation_shap_values.npy"
)

print("TranAD SHAP values:", TranADSHAPValues.shape)

TranAD SHAP values: (20, 1000, 12)


In [33]:
# Create the same TranAD Anomaly Score Wrapper for TranAD that was used during SHAP explanations

TranADScoreWrapper = TranADAnomalyScoreWrapper(TranAD)

TranADScoreWrapper.eval()

# Create the same Explainer
TranADSHAPExplainer = shap.GradientExplainer(
    TranADScoreWrapper,
    background_ECGs,
    batch_size=1,
)

In [34]:
import gc

TranADSHAPOriginalImportanceValues = []
TranADSHAPNoisyImportanceValues = []

for position, sample in zip(SHAPStabilityPositions, SHAPStabilitySamples):
    ecg = sample["ecg"].to(TranAD.Device)

    original_shap_values = TranADSHAPValues[position]
    original_importance = np.abs(original_shap_values).sum(axis=1)

    noisy_importances = []

    for _ in range(num_noisy_variants):

        # create a noisy variant and calculate it's shap values
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)
        noisy_shap_values = TranADSHAPExplainer.shap_values(noisy_ecg)

        if isinstance(noisy_shap_values, list):
            noisy_shap_values = noisy_shap_values[0]

        noisy_shap_values = np.asarray(noisy_shap_values)

        if noisy_shap_values.ndim == 4:
            noisy_shap_values = noisy_shap_values[..., 0]

        noisy_shap_values = noisy_shap_values[0]

        noisy_importance = np.abs(noisy_shap_values).sum(axis=1)

        noisy_importances.append(noisy_importance)
        # Clear CUDA cache after each noisy variant calculation to free up memory
        if torch.cuda.is_available():
            del noisy_ecg # Explicitly delete tensors to aid GC
            del noisy_shap_values
            gc.collect() # Trigger garbage collection
            torch.cuda.empty_cache()

    TranADSHAPOriginalImportanceValues.append(original_importance)
    TranADSHAPNoisyImportanceValues.append(noisy_importances)

    print(
        "Computed TranAD SHAP stability importances for index:",
        sample["index"],
        "case:",
        sample["case_type"]
    )
    # Clear CUDA cache after processing each sample as well
    if torch.cuda.is_available():
        gc.collect() # Trigger garbage collection
        torch.cuda.empty_cache()

Computed TranAD SHAP stability importances for index: 2154 case: TN
Computed TranAD SHAP stability importances for index: 1647 case: TN
Computed TranAD SHAP stability importances for index: 161 case: TN
Computed TranAD SHAP stability importances for index: 1394 case: TN
Computed TranAD SHAP stability importances for index: 929 case: TN
Computed TranAD SHAP stability importances for index: 1134 case: TP
Computed TranAD SHAP stability importances for index: 1854 case: TP
Computed TranAD SHAP stability importances for index: 102 case: TP
Computed TranAD SHAP stability importances for index: 1310 case: TP
Computed TranAD SHAP stability importances for index: 1462 case: TP


In [35]:
# Evaluate

TranADSHAPStabilityList = (
    StabilityEvaluator.evaluate_samples(
        samples=SHAPStabilitySamples,
        original_importance_values=TranADSHAPOriginalImportanceValues,
        noisy_importance_values=TranADSHAPNoisyImportanceValues,
        model_name="TranAD",
        explanation_name="SHAP",
    )
)

TranADSHAPStabilityResults = pd.DataFrame(
    TranADSHAPStabilityList
)

TranADSHAPStabilityResults

,Model,Explanation,Index,Case Type,Label,Stability Score,Stability Std,Number of Noisy Variants
0,TranAD,SHAP,2154,TN,0,0.998008,0.000252,2
1,TranAD,SHAP,1647,TN,0,0.995479,0.000016,2
2,TranAD,SHAP,161,TN,0,0.997442,0.000713,2
3,TranAD,SHAP,1394,TN,0,0.996616,0.001444,2
4,TranAD,SHAP,929,TN,0,0.990113,0.004601,2
5,TranAD,SHAP,1134,TP,1,0.998372,0.000112,2
6,TranAD,SHAP,1854,TP,1,0.998817,0.000384,2
7,TranAD,SHAP,102,TP,1,0.999116,0.000277,2
8,TranAD,SHAP,1310,TP,1,0.999350,0.000011,2
9,TranAD,SHAP,1462,TP,1,0.999311,0.000134,2


In [36]:
TranADSHAPStabilitySummary = (
    TranADSHAPStabilityResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADSHAPStabilitySummary

,Model,Explanation,Case Type,mean,std,count
0,TranAD,SHAP,TN,0.995531,0.003175,5
1,TranAD,SHAP,TP,0.998993,0.000406,5


In [37]:
# Save

TranADSHAPStabilityResults.to_csv(
    STABILITY_DIR / "tranad_shap_stability_results.csv",
    index=False,
)

TranADSHAPStabilitySummary.to_csv(
    STABILITY_DIR / "tranad_shap_stability_summary.csv",
    index=False,
)

# Final Stability Evaluation Results

In [38]:
StabilitySummary = pd.concat(
    [
        AnoFormerAttentionStabilitySummary,
        AnoFormerSHAPStabilitySummary,
        TranADAttentionStabilitySummary,
        TranADSHAPStabilitySummary,
    ],
    ignore_index=True,
)

StabilitySummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,Attention,TN,0.999995,2.326993e-06,10
1,AnoFormer,Attention,TP,0.999946,1.004537e-04,10
2,AnoFormer,SHAP,TN,0.993175,3.865145e-03,5
3,AnoFormer,SHAP,TP,0.984165,1.303369e-02,5
4,TranAD,Attention,TN,1.000000,6.505313e-16,10
5,TranAD,Attention,TP,1.000000,7.797950e-16,10
6,TranAD,SHAP,TN,0.995531,3.174912e-03,5
7,TranAD,SHAP,TP,0.998993,4.061982e-04,5


In [39]:
StabilitySummary.to_csv(
    STABILITY_DIR / "reconstruction_error_agreement_summary_all_methods.csv",
    index=False,
)